In [ ]:
from pathlib import Path
import sys

# This notebook lives in kuramoto/LMSSPP/notebooks.
# Add the local LMSSPP source tree without requiring package installation.
LMSSPP_SRC = Path("../src").resolve()
if not LMSSPP_SRC.exists():
    # Fallback for running the notebook from the repository root.
    LMSSPP_SRC = Path("pitch-website/public/notebooks/kuramoto/LMSSPP/src").resolve()
if str(LMSSPP_SRC) not in sys.path:
    sys.path.insert(0, str(LMSSPP_SRC))

print("LMSSPP source:", LMSSPP_SRC)


Interesting phenomena is reproduced with this settings:
```python
config = SimulationConfig(
    n_fibers=50,
    alpha=0.99,
    n_per_fiber=200,
    grid_size=512,
    domain_radius=10.0,
    make_animation=True,
    backend="numpy",
    integrator="fixed_rk2",
)
```

# The Peszek-Poyato alignment dynamic

In [ ]:
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_dynamics_widget

                                # grid_size=256, animation_density_grid_size=512,
                                # dt=0.0002,
                                # animation_frame_duration_ms=90,
                                # domain_radius=15.,

config = SimulationConfig(n_fibers=10,alpha=0.99, n_per_fiber=100, 
                    color_scheme="phase_color",
                                max_steps=1000, make_animation=True)
                                
widget = make_dynamics_widget(config,
                                second_panel="velocity",)
widget                


# Barycenter and Critical Point per-fiber dynamics

In [ ]:
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_dynamics_widget

                                # grid_size=256, animation_density_grid_size=512,
                                # dt=0.0002,
                                # animation_frame_duration_ms=90,

config = SimulationConfig(n_fibers=10,alpha=0.985, n_per_fiber=300, 
                    integrator="adaptive_rk4",
                                domain_radius=8.,
                                max_steps=500, 
                    color_scheme="phase_color", make_animation=True)
                    
                                
widget = make_dynamics_widget(config,
                                second_panel="barycenters")
widget                


In [ ]:
K=2.278e-02;K=2.277e-02

# Non-Peszek-Poyato dynamics discovered accidentally through fixed RK2 integration with $\alpha\gg 0$, Here: $\alpha=0.99$

In [ ]:
config_interesting = SimulationConfig(
    n_fibers=10,
    alpha=0.99,
    n_per_fiber=100,
    grid_size=256,
    domain_radius=5.0,
    make_animation=True,
    integrator="fixed_rk2",
    max_steps=1000,
)
widget = make_dynamics_widget(config_interesting, 
                                second_panel="velocity",
                                )
widget                

# Predictive PP systems

In [ ]:
from dataclasses import replace
from lmsspp.dynamics.pp_cs_equilibria import make_finite_horizon_gauge_averaged_widget

predictive_pp_base_config = SimulationConfig(
    n_fibers=10,
    n_per_fiber=100,
    alpha=0.99,
    K=None,
    grid_size=128,
    domain_radius=6.0,
    dt=0.01,
    dt_min=1.0e-4,
    dt_max=0.02,
    max_steps=1500,
    tol_rms=0.0,
    max_displacement_per_step=0.5,
    integrator="adaptive_rk2",
    prediction_horizon_tau=0.055,
    color_scheme="phase_color",
    make_animation=True,
    record_every=10,
)

averaged_predictive_pp_widget = make_finite_horizon_gauge_averaged_widget(
    replace(predictive_pp_base_config, 
            predictive_pp_weight=0.5,
    ),
    second_panel="velocity",
)
averaged_predictive_pp_widget

In [ ]:
pure_predictive_pp_widget = make_finite_horizon_gauge_averaged_widget(
    replace(predictive_pp_base_config, predictive_pp_weight=1),
    second_panel="velocity",
)

pure_predictive_pp_widget


# ALl things below are very non-standard dynamics that share the core "Potential energy+interaction" core of the PP systems but do some very non-standard break of the orginal formulation

Treat these as exploratory research implementations:

# Work in progress non quadratic hamiltonian dynamic

In [ ]:
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_hamiltonian_exponent_widget

config=SimulationConfig(n_fibers=10,n_per_fiber=16,grid_size=128,
domain_radius=6.0,max_steps=400,make_animation=True, integrator="adaptive_rk4",
hamiltonian_q=2.0,hamiltonian_epsH=0,initialization_algorithm='raw')
make_hamiltonian_exponent_widget(config)

# The Projective perturbation of the $\omega\cdot x$ natural parameter pairing

In [ ]:
from lmsspp.dynamics.pp_cs_equilibria import ProjectivePeszekPoyatoDynamicsWidget, SimulationConfig

config = SimulationConfig(n_fibers=10,alpha=0.99, n_per_fiber=100, 
                                # grid_size=256, animation_density_grid_size=512,
                                # dt=0.0002,
                                # animation_frame_duration_ms=90,
                                domain_radius=5.,
                                max_steps=1000, make_animation=True)
w = ProjectivePeszekPoyatoDynamicsWidget(config)
w 

# The continuous density limit and entropic $W_{2,\nu}$ diffusion

In [ ]:
# Continuous-density backend (explicit FV on grid fiber densities r_k)
from lmsspp.dynamics.pp_cs_equilibria import (
    DENSITY_DIAGNOSTIC_FIELDS,
    SimulationConfig,
    make_density_initial_condition,
    run_density_simulation,
)

density_config = SimulationConfig(
    n_fibers=6,
    alpha=0.5,
    K=1.0,
    eps_entropy=0.02,
    grid_size=64,
    domain_radius=6.0,
    dt=0.01,
    max_steps=120,
    make_animation=False,
    record_free_energy=True,
    record_entropy_balance=True,
    density_solver="explicit_fv",
    seed=2026,
)
initial = make_density_initial_condition(density_config)
result = run_density_simulation(density_config, initial)

print(f"steps={result.steps}, runtime={result.runtime_seconds:.2f}s")
if result.diagnostics.size:
    last = {name: result.diagnostics[-1, i] for i, name in enumerate(DENSITY_DIAGNOSTIC_FIELDS)}
    print(f"final rms velocity={last['rms_velocity']:.4g}")
    for key in ("total_mass", "free_energy", "entropy_H", "trace_div_A", "fisher_information"):
        print(f"{key}={last[key]:.4g}")
result

In [ ]:
# Continuous-density widget (heatmap playback; click Precompute, then Play/Step/slider)
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_continuous_density_widget

density_widget_config = SimulationConfig(
    n_fibers=8,
    alpha=0.8,
    K=1.0,
    eps_entropy=0.03,
    grid_size=128,
    
    domain_radius=8.0,
    dt=0.008,
    max_steps=300,

    
    make_animation=True,
    trajectory_frame_count=0,
    record_free_energy=True,
    record_entropy_balance=True,
    density_solver="split_implicit_diffusion",
    seed=2026,
)
density_widget = make_continuous_density_widget(density_widget_config)
density_widget